# P-score de checkpoints (`notebooks/checkpoints`)

Calcula el **parallelism score (p-score)** de cada checkpoint guardado en esta carpeta, sobre los
splits de **validación** y **test** del dataset correspondiente.

Puntos clave del diseño:

* **Reutiliza la misma función que el entrenamiento**: `visgen.trainers.representation_metrics.compute_representation_metrics_on_loader`,
  la misma que llama `BaseTrainer.train` (`visgen/trainers/trainer.py`). No se reimplementa ninguna métrica.
* **Reproduce exactamente los mismos splits que el entrenamiento**: se reconstruye la configuración igual
  que `main.py` (merge de `base.yml` + model cfg + data cfg + experiment cfg + overrides de CLI del runner),
  se llama a `fix_random(cfg.seed)` y luego a `get_dataloaders(cfg.data, writer, seed=cfg.seed)`.
  Como `random_split` usa un `torch.Generator` sembrado con `seed + índice_del_split` y `ood_validation_split`
  consume el estado global de `random` fijado por `fix_random`, los subconjuntos son bit-a-bit los mismos.
* **Funciona con y sin GPU**: el dispositivo se elige con `torch.cuda.is_available()` y los checkpoints se
  cargan con `map_location=device`.

> El nombre del archivo del checkpoint indica el dataset y la arquitectura (por defecto se asume
> `split_mixer algebraic` → `configs/models/split_resnet_algebraic_non_iid.yml`). Revisa la celda de
> **Configuración** y la tabla `MANUAL_SPECS` si algún archivo no se parsea bien.

## 1. Entorno e imports

In [ ]:
import os
import re
import sys
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
import torch
from omegaconf import OmegaConf


def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "main.py").exists() and (candidate / "visgen").is_dir():
            return candidate
    raise RuntimeError(f"No se encontró la raíz del repo partiendo de {start}")


# El notebook vive en <repo>/notebooks/checkpoints
NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = _find_repo_root(NOTEBOOK_DIR)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Los configs usan rutas relativas a la raíz del repo (p.ej. data/dsprites/dsprites.npz),
# igual que cuando se ejecuta `python main.py` desde ahí.
os.chdir(REPO_ROOT)

from visgen.datasets import get_dataloaders  # noqa: E402
from visgen.models import get_model  # noqa: E402
from visgen.trainers.representation_metrics import (  # noqa: E402
    compute_representation_metrics_on_loader,
)
from visgen.utils.general import BaseLogger, fix_random, register_resolvers  # noqa: E402

try:
    register_resolvers()
except Exception as exc:  # el resolver ya está registrado si se re-ejecuta la celda
    print(f"[info] register_resolvers(): {exc}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("repo root :", REPO_ROOT)
print("notebook  :", NOTEBOOK_DIR)
print("device    :", DEVICE)
print("torch     :", torch.__version__)

## 2. Configuración

Todo lo ajustable está en esta celda.

In [ ]:
# --- Dónde buscar los checkpoints -------------------------------------------------
CHECKPOINT_DIR = NOTEBOOK_DIR
CHECKPOINT_GLOBS = ("*.pth.tar", "*.pth", "*.pt")

# --- Qué splits evaluar -----------------------------------------------------------
# Nombres tal como los devuelve get_dataloaders(). "validation" y "testing" son los que
# el trainer usa como val_loader / test_loader.
# Si quieres además el split sobre el que el trainer loguea las métricas de representación,
# agrega "val_4cases_raw" (o "val_4cases" si no hay wrapper non-iid).
SPLITS_TO_EVALUATE = ("validation", "testing")

# --- Config por defecto para reconstruir el experimento ---------------------------
# Corresponde a ain_algebraic_runner.sh: split_mixer con mixer algebraico.
DEFAULT_MODEL_CFG = "configs/models/split_resnet_algebraic_non_iid.yml"
DEFAULT_EXPERIMENT_CFG = "configs/experiments/metrics.yml"
DATA_CFG_TEMPLATE = "configs/datasets/{dataset}_non_iid.yml"
BASE_CFG = "configs/base.yml"

# Semilla por dataset a usar cuando el nombre del archivo no la incluye.
#
# Son las semillas con mayor test_acc para arch="split_resnet_algebraic_non_iid" (c=1),
# leídas de los resultados ya parseados en
#   /home/araymond/storage/investigacion/licg/scalable-compositional-generalization/{dataset}_id.pkl
# (salida de out/parse_out.py --selection id). Ver la celda "Procedencia de las semillas".
#
# OJO: la semilla determina los splits (random_split + holdout OOD). Si un checkpoint no
# corresponde al mejor run, pon la semilla en el nombre del archivo (p.ej. ..._seed3.pth.tar)
# o decláralala en MANUAL_SPECS.
BEST_SEED_BY_DATASET = {
    "dsprites": 1,   # test_acc 71.49
    "iraven": 2,     # test_acc 80.02
    "cars3d": 5,     # test_acc 56.82
    "shapes3d": 4,   # test_acc 94.99
    "clevr": 1,      # test_acc 65.00
    "mpi3d": 2,      # test_acc 68.07
}

# Último recurso si el dataset tampoco está en la tabla de arriba.
DEFAULT_SEED = 1

# --- Parámetros de las métricas de representación ---------------------------------
# Por defecto se leen de cfg.training.representation_metrics (igual que el trainer).
# Estos overrides sólo se aplican si no son None.
#
# ATENCIÓN: topsim/twonn son O(n^2) en memoria sobre las muestras "pairwise". En el
# entrenamiento el split val_4cases es chico, pero el test set puede tener cientos de
# miles de ejemplos. Si te quedas sin memoria, fija PAIRWISE_MAX_SAMPLES (p.ej. 3000).
# El p-score se calcula sobre ese mismo subconjunto, así que cámbialo sólo si aceptas
# que deja de ser idéntico al cálculo con el split completo.
MAX_SAMPLES_OVERRIDE: Optional[int] = None           # nº de ejemplos leídos del loader
PAIRWISE_MAX_SAMPLES_OVERRIDE: Optional[int] = None  # submuestreo para métricas pairwise

# --- DataLoader -------------------------------------------------------------------
# No afecta a los splits (no hay shuffle en get_dataloaders), sólo a la velocidad.
NUM_WORKERS = {"training": 0, "testing": 4}

# --- Overrides de CLI del runner (ain_algebraic_runner.sh) ------------------------
# Se aplican como dotlist, exactamente igual que los argumentos sueltos de main.py.
EXTRA_CLI_OVERRIDES: List[str] = []

# --- Dónde guardar los resultados -------------------------------------------------
RESULTS_CSV = CHECKPOINT_DIR / "pscore_results.csv"

## 3. Tabla de splits por dataset

Copiada de `ain_algebraic_runner.sh` (y compartida con `ain_alg_lambda_runner.sh`,
`split_mixer_runner.sh`, etc.). Son los overrides de CLI que el runner pasa a `main.py`
y que definen el split de composición usado en entrenamiento.

In [ ]:
DATASET_SPLIT_SPECS = {
    "dsprites": {
        "c": 1,
        "attr_difficulty": "[2,3,14,14]",
        "split_attributes": "scale_shape_x-position_y-position",
    },
    "iraven": {
        "c": 1,
        "attr_difficulty": "[6,3,3]",
        "split_attributes": "size_type_color",
    },
    "cars3d": {
        "c": 1,
        "attr_difficulty": "[15,2,113]",
        "split_attributes": "elevation_type_orientation",
    },
    "shapes3d": {
        "c": 1,
        "attr_difficulty": "[7,7,7,6,3]",
        "split_attributes": "wall_floor_object_scale_shape",
    },
    "clevr": {
        "c": 1,
        "attr_difficulty": "[2,2,1,7]",
        "split_attributes": "shape_size_material_color",
    },
    "mpi3d": {
        "c": 1,
        "attr_difficulty": "[5,4,2,2,34,34]",
        "split_attributes": "color_shape_height_bgcolor_x-axis_y-axis",
    },
}

SPLIT_TYPE = "general_composition"
KNOWN_DATASETS = tuple(DATASET_SPLIT_SPECS)

### Procedencia de las semillas

`BEST_SEED_BY_DATASET` sale de los resultados ya parseados (`out/parse_out.py --selection id`),
que viven en otra copia del repo:

```
/home/araymond/storage/investigacion/licg/scalable-compositional-generalization/{dataset}_id.pkl
```

Para cada dataset se tomó la semilla con mayor `test_acc` entre las 5 corridas de
`arch == "split_resnet_algebraic_non_iid"` con `c == 1`. La celda siguiente recalcula la tabla
para que puedas verificarla (se salta sin error si esa ruta no está montada).

In [ ]:
RESULTS_PKL_DIR = Path("/home/araymond/storage/investigacion/licg/scalable-compositional-generalization")
RESULTS_ARCH = "split_resnet_algebraic_non_iid"


def best_seeds_from_results(arch: str = RESULTS_ARCH, c: int = 1, metric: str = "test_acc"):
    """Recalcula la mejor semilla por dataset desde los {dataset}_id.pkl."""
    if not RESULTS_PKL_DIR.is_dir():
        print(f"[info] {RESULTS_PKL_DIR} no está disponible; se usa BEST_SEED_BY_DATASET tal cual.")
        return None

    rows = []
    for dataset in KNOWN_DATASETS:
        pkl = RESULTS_PKL_DIR / f"{dataset}_id.pkl"
        if not pkl.exists():
            print(f"[info] falta {pkl.name}")
            continue
        df = pd.read_pickle(pkl)
        sub = df[df["arch"] == arch].copy()
        if sub.empty:
            print(f"[info] {dataset}: sin corridas de {arch}")
            continue
        sub["seed"] = sub["seed"].astype(int)
        sub["c"] = sub["c"].astype(int)
        sub = sub[sub["c"] == c]
        if sub.empty:
            continue
        best = sub.loc[sub[metric].idxmax()]
        rows.append(
            {
                "dataset": dataset,
                "c": int(best["c"]),
                "best_seed": int(best["seed"]),
                metric: best[metric],
                "val_acc": best["val_acc"],
                "n_runs": len(sub),
                "coincide_con_tabla": int(best["seed"]) == BEST_SEED_BY_DATASET.get(dataset),
            }
        )
    return pd.DataFrame(rows)


_best_seeds = best_seeds_from_results()
_best_seeds

## 4. Parseo de los nombres de checkpoint

El dataset se detecta por substring; la arquitectura por palabras clave; la semilla por
un sufijo tipo `seed3` / `_s3` / `_3`. Cualquier archivo que no se parsee bien se puede
declarar a mano en `MANUAL_SPECS`.

In [ ]:
# Palabras clave -> config de modelo. Se evalúan en orden (la primera que calce gana),
# por eso las más específicas van primero.
MODEL_CFG_KEYWORDS = (
    ("no_mixer", "configs/models/split_resnet_mixer_no_mixer.yml"),
    ("algebraic", "configs/models/split_resnet_algebraic_non_iid.yml"),
    ("_alg", "configs/models/split_resnet_algebraic_non_iid.yml"),
    ("split_mixer", "configs/models/split_resnet_mixer.yml"),
    ("split_resnet_mixer", "configs/models/split_resnet_mixer.yml"),
)

# Overrides explícitos por nombre de archivo (sin ruta). Cualquier clave de CheckpointSpec
# es válida, p.ej.:
# MANUAL_SPECS = {
#     "dsprites_split_mixer_algebraic.pth.tar": {"seed": 3, "experiment_cfg": "configs/experiments/comp_ablation.yml"},
# }
MANUAL_SPECS: Dict[str, Dict[str, Any]] = {}

_SEED_PATTERNS = (
    re.compile(r"seed[-_]?(\d+)"),
    re.compile(r"(?:^|[-_])s(\d+)(?:[-_]|$)"),
    re.compile(r"[-_](\d+)$"),
)


@dataclass
class CheckpointSpec:
    path: Path
    dataset: str
    seed: int
    model_cfg: str
    experiment_cfg: str = DEFAULT_EXPERIMENT_CFG
    data_cfg: Optional[str] = None          # por defecto DATA_CFG_TEMPLATE.format(...)
    split: str = SPLIT_TYPE
    overrides: List[str] = field(default_factory=list)

    @property
    def name(self) -> str:
        return self.path.name

    def resolved_data_cfg(self) -> str:
        return self.data_cfg or DATA_CFG_TEMPLATE.format(dataset=self.dataset)


def _strip_extensions(name: str) -> str:
    for suffix in (".pth.tar", ".pth", ".pt"):
        if name.endswith(suffix):
            return name[: -len(suffix)]
    return name


def _detect_dataset(stem: str) -> Optional[str]:
    matches = [ds for ds in KNOWN_DATASETS if ds in stem]
    if not matches:
        return None
    # el nombre más largo gana, para no confundir prefijos
    return max(matches, key=len)


def _detect_model_cfg(stem: str) -> str:
    for keyword, cfg_path in MODEL_CFG_KEYWORDS:
        if keyword in stem:
            return cfg_path
    return DEFAULT_MODEL_CFG


def _detect_seed(stem: str) -> Optional[int]:
    for pattern in _SEED_PATTERNS:
        match = pattern.search(stem)
        if match:
            return int(match.group(1))
    return None


def parse_checkpoint(path: Path) -> CheckpointSpec:
    stem = _strip_extensions(path.name).lower()
    manual = dict(MANUAL_SPECS.get(path.name, {}))

    dataset = manual.pop("dataset", None) or _detect_dataset(stem)
    if dataset is None:
        raise ValueError(
            f"No pude inferir el dataset de '{path.name}'. "
            f"Datasets conocidos: {KNOWN_DATASETS}. Decláralo en MANUAL_SPECS."
        )

    seed = manual.pop("seed", None)
    if seed is None:
        seed = _detect_seed(stem)
    if seed is None:
        seed = BEST_SEED_BY_DATASET.get(dataset, DEFAULT_SEED)
        print(
            f"[warn] '{path.name}': sin semilla en el nombre, uso la mejor semilla "
            f"conocida para {dataset}: {seed}"
        )

    model_cfg = manual.pop("model_cfg", None) or _detect_model_cfg(stem)

    return CheckpointSpec(
        path=path,
        dataset=dataset,
        seed=int(seed),
        model_cfg=model_cfg,
        **manual,
    )


def discover_checkpoints(directory: Path = CHECKPOINT_DIR) -> List[CheckpointSpec]:
    paths = sorted({p for pattern in CHECKPOINT_GLOBS for p in directory.glob(pattern)})
    specs = []
    for path in paths:
        try:
            specs.append(parse_checkpoint(path))
        except ValueError as exc:
            print(f"[skip] {exc}")
    return specs


CHECKPOINTS = discover_checkpoints()
if not CHECKPOINTS:
    print(f"[warn] No hay checkpoints en {CHECKPOINT_DIR} (patrones: {CHECKPOINT_GLOBS})")
pd.DataFrame(
    [
        {
            "archivo": spec.name,
            "dataset": spec.dataset,
            "seed": spec.seed,
            "model_cfg": spec.model_cfg,
            "data_cfg": spec.resolved_data_cfg(),
            "experiment_cfg": spec.experiment_cfg,
        }
        for spec in CHECKPOINTS
    ]
)

## 5. Reconstrucción de la configuración (igual que `main.py`)

`custom_cfg_conflict_resolution` y la composición de `cfg.path.full` se replican tal cual
para que el `cfg` resultante sea el mismo que vio el entrenamiento (batch sizes incluidos,
que afectan cómo se recorre el loader).

In [ ]:
def _custom_cfg_conflict_resolution(cfg, model_cfg, data_cfg, experiment_cfg, cfg_cli):
    """Copia de main.custom_cfg_conflict_resolution."""
    if (
        hasattr(model_cfg, "data")
        and hasattr(model_cfg.data.training, "batch_size")
        and hasattr(data_cfg.data.training, "batch_size")
    ):
        cfg.data.training.batch_size = min(
            data_cfg.data.training.batch_size, model_cfg.data.training.batch_size
        )
        cfg.data.testing.batch_size = min(
            data_cfg.data.testing.batch_size, model_cfg.data.testing.batch_size
        )
    if (
        hasattr(experiment_cfg, "training")
        and hasattr(data_cfg, "training")
        and not (hasattr(cfg_cli, "training") and hasattr(cfg_cli.training, "n_epoch"))
    ):
        cfg.training.n_epoch = data_cfg.training.n_epoch
    if hasattr(model_cfg, "model") and hasattr(model_cfg.model, "preprocessing"):
        cfg.model.preprocessing = model_cfg.model.preprocessing
    return cfg


def cli_overrides_for(spec: CheckpointSpec) -> List[str]:
    """Los mismos argumentos sueltos que pasa ain_algebraic_runner.sh a main.py."""
    ds_spec = DATASET_SPLIT_SPECS[spec.dataset]
    split_attributes = ds_spec["split_attributes"]
    overrides = [
        f"data.training.targets={split_attributes}",
        f"data.training.split_attributes={split_attributes}",
        f"data.training.split={spec.split}",
        f"data.testing.split={spec.split}",
        f"data.training.c={ds_spec['c']}",
        f"data.testing.c={ds_spec['c']}",
        f"data.training.attr_difficulty={ds_spec['attr_difficulty']}",
        f"data.testing.attr_difficulty={ds_spec['attr_difficulty']}",
        f"seed={spec.seed}",
        f"data.training.num_workers={NUM_WORKERS['training']}",
        f"data.testing.num_workers={NUM_WORKERS['testing']}",
        "logger.name=base",
    ]
    overrides += EXTRA_CLI_OVERRIDES
    overrides += spec.overrides
    return overrides


def build_cfg(spec: CheckpointSpec):
    cfg_base = OmegaConf.load(REPO_ROOT / BASE_CFG)
    cfg_data = OmegaConf.load(REPO_ROOT / spec.resolved_data_cfg())
    cfg_model = OmegaConf.load(REPO_ROOT / spec.model_cfg)
    cfg_experiment = OmegaConf.load(REPO_ROOT / spec.experiment_cfg)
    cfg_cli = OmegaConf.from_dotlist(cli_overrides_for(spec))

    # El orden del merge define la jerarquía; es el mismo de main.py.
    cfg = OmegaConf.merge(cfg_base, cfg_model, cfg_data, cfg_experiment, cfg_cli)
    cfg = _custom_cfg_conflict_resolution(cfg, cfg_model, cfg_data, cfg_experiment, cfg_cli)

    cfg["device"] = str(DEVICE)
    cfg.logger["sweep"] = False
    cfg.logger["run_id"] = "eval"

    # Composición de la ruta de resultados (main.py). Algunos modelos (p.ej. ed) leen
    # cfg.model.path para localizar codebooks, así que se replica.
    run_name = Path(spec.experiment_cfg).stem
    model_name = Path(spec.model_cfg).stem
    model_groupby = cfg.data.training.targets
    split_value = cfg.data.training.get("c")
    if split_value is None:
        split_value = cfg.data.training.get("split_difficulty")
    split_label = (
        f"{cfg.data.training.split}_{split_value}"
        if split_value is not None
        else cfg.data.training.split
    )
    cfg.model.path = cfg.path.full = os.path.join(
        cfg.path.base, run_name, cfg.data.training.dataset,
        split_label, model_name, model_groupby, str(cfg.seed),
    )
    return cfg

### Semillas

En el repo hay **una sola semilla de nivel de experimento**, `cfg.seed` (el `--seed=$seed` de los
runners). De ella se derivan todas las semillas de datos, más dos constantes que **no** dependen
del run:

| Semilla | Valor | Dónde se usa |
| --- | --- | --- |
| `cfg.seed` | `--seed=$seed` | `fix_random()` — estado global de `random`, del que depende el holdout OOD (`ood_validation_split` usa `random.sample`) |
| `cfg.seed + i` | `i` = índice del split en `cfg.data` (`training`→+0, `testing`→+1) | generador de `random_split` train/val y `worker_init_fn` ([datasets/__init__.py:251](../../visgen/datasets/__init__.py)) |
| `data.*.non_iid.seed` | `${seed}` → interpolado a `cfg.seed` | muestreo de pares del `NonIIDWrapper` sobre `training` |
| `non_iid.split_overrides.val_4cases.seed` | `123`, fijo | wrapper determinista de `val_4cases` (igual para todas las semillas) |
| `training.representation_metrics.sampling_seed` | `0`, fijo | submuestreo `pairwise_max_samples` dentro del cálculo de métricas |

O sea: basta con acertarle a `cfg.seed` para reproducir los splits; el resto se deriva solo vía
interpolación de OmegaConf. Si tu corrida usó valores distintos para las constantes, pásalos en
`spec.overrides` (p.ej. `"data.training.non_iid.split_overrides.val_4cases.seed=123"`).

`validation` y `testing` **no** llevan wrapper non-iid con estos configs (`apply_to: ["training", "val_4cases"]`),
así que sobre esos dos splits el p-score no depende de las semillas del wrapper — sólo de `cfg.seed`
a través de `random_split` y del holdout OOD.

In [ ]:
def describe_seeds(cfg) -> Dict[str, Any]:
    """Semillas efectivas derivadas de cfg.seed (para dejar registro en los resultados)."""
    info = {
        "seed": int(cfg.seed),
        "random_split_seed_training": int(cfg.seed),      # cfg.seed + índice 0
        "random_split_seed_testing": int(cfg.seed) + 1,   # cfg.seed + índice 1
        "non_iid_seed": None,
        "val_4cases_non_iid_seed": None,
        "rep_sampling_seed": cfg.training.get("representation_metrics", {}).get("sampling_seed", 0),
    }
    non_iid = cfg.data.training.get("non_iid")
    if non_iid is not None and not isinstance(non_iid, str):
        info["non_iid_seed"] = non_iid.get("seed")
        overrides = non_iid.get("split_overrides", {}) or {}
        if "val_4cases" in overrides:
            info["val_4cases_non_iid_seed"] = overrides["val_4cases"].get("seed", non_iid.get("seed"))
    return info

## 6. Dataloaders con los splits exactos del entrenamiento

`fix_random(cfg.seed)` + `get_dataloaders(cfg.data, writer, seed=cfg.seed)`, en ese orden y
sin consumir aleatoriedad entremedio — igual que `main.py`. El writer es un `BaseLogger`
(no-op) porque `get_dataloaders` loguea el conteo de valores por atributo.

Los loaders se cachean por (dataset, seed, split, dificultad, targets, batch sizes) para no
reconstruir el dataset por cada checkpoint.

In [ ]:
_DATALOADER_CACHE: Dict[str, Dict[str, Any]] = {}


def _dataloader_cache_key(cfg) -> str:
    return OmegaConf.to_yaml(
        OmegaConf.create({"seed": cfg.seed, "data": cfg.data}), resolve=True
    )


def build_dataloaders(cfg):
    key = _dataloader_cache_key(cfg)
    if key in _DATALOADER_CACHE:
        return _DATALOADER_CACHE[key]

    # Mismo orden que main.py: fix_random y acto seguido get_dataloaders.
    fix_random(cfg.seed)
    loaders = get_dataloaders(cfg.data, BaseLogger(), seed=cfg.seed)

    print("  splits disponibles:", ", ".join(f"{k} ({len(v.dataset)})" for k, v in loaders.items()))
    _DATALOADER_CACHE[key] = loaders
    return loaders

## 7. Carga del modelo desde el checkpoint

In [ ]:
def load_state_dict_from(path: Path) -> Dict[str, Any]:
    try:
        return torch.load(path, map_location=DEVICE)
    except Exception:
        # torch >= 2.6 usa weights_only=True por defecto; los checkpoints guardan
        # además best_ams (dict de floats), que puede requerir el unpickler completo.
        return torch.load(path, map_location=DEVICE, weights_only=False)


def load_model(cfg, spec: CheckpointSpec):
    model = get_model(cfg).to(DEVICE)
    ckpt = load_state_dict_from(spec.path)
    model.load_state_dict(ckpt["model_state_dict"], strict=True)
    model.eval()
    return model, ckpt

## 8. Cálculo del p-score

Se llama a `compute_representation_metrics_on_loader`, la misma función que invoca
`BaseTrainer.train`. Los parámetros salen de `cfg.training.representation_metrics`
(mismo parseo que el trainer, incluido el caso en que sea un booleano).

In [ ]:
def rep_metrics_params(cfg) -> Dict[str, Any]:
    """Mismo parseo que BaseTrainer.train sobre cfg.training.representation_metrics."""
    rep_cfg = cfg.training.get("representation_metrics", {})
    if isinstance(rep_cfg, bool):
        rep_cfg = {"enabled": rep_cfg}
    params = {
        "max_samples": rep_cfg.get("max_samples"),
        "pairwise_max_samples": rep_cfg.get("pairwise_max_samples"),
        "sampling_seed": rep_cfg.get("sampling_seed", 0),
        "variance_threshold": float(rep_cfg.get("variance_threshold", 0.9)),
        "observed_metric": rep_cfg.get("observed_metric", "cosine"),
    }
    if MAX_SAMPLES_OVERRIDE is not None:
        params["max_samples"] = MAX_SAMPLES_OVERRIDE
    if PAIRWISE_MAX_SAMPLES_OVERRIDE is not None:
        params["pairwise_max_samples"] = PAIRWISE_MAX_SAMPLES_OVERRIDE
    return params


@torch.no_grad()
def compute_pscore(model, loader, params: Dict[str, Any]) -> Optional[Dict[str, float]]:
    return compute_representation_metrics_on_loader(
        model=model, loader=loader, device=DEVICE, **params
    )

## 9. Evaluación

Recorre los checkpoints y calcula el p-score en cada split pedido.

In [ ]:
from tqdm.auto import tqdm  # noqa: E402

rows: List[Dict[str, Any]] = []
failures: List[Dict[str, str]] = []

for spec in tqdm(CHECKPOINTS, desc="checkpoints"):
    print(f"\n=== {spec.name} | dataset={spec.dataset} seed={spec.seed} "
          f"model={Path(spec.model_cfg).stem} ===")
    try:
        cfg = build_cfg(spec)
        seeds = describe_seeds(cfg)
        print("  semillas:", seeds)
        loaders = build_dataloaders(cfg)
        model, ckpt = load_model(cfg, spec)
        params = rep_metrics_params(cfg)
        print("  params rep-metrics:", params)
    except Exception as exc:
        print(f"  [error] {type(exc).__name__}: {exc}")
        failures.append({"archivo": spec.name, "split": "-", "error": f"{type(exc).__name__}: {exc}"})
        continue

    epoch = ckpt.get("epoch")
    best_ams = ckpt.get("best_ams") or {}

    for split in SPLITS_TO_EVALUATE:
        loader = loaders.get(split)
        if loader is None:
            print(f"  [skip] split '{split}' no existe (disponibles: {sorted(loaders)})")
            failures.append({"archivo": spec.name, "split": split, "error": "split inexistente"})
            continue
        try:
            metrics = compute_pscore(model, loader, params)
        except Exception as exc:
            print(f"  [error] split '{split}': {type(exc).__name__}: {exc}")
            failures.append({"archivo": spec.name, "split": split, "error": f"{type(exc).__name__}: {exc}"})
            continue
        if metrics is None:
            print(f"  [skip] split '{split}' vacío")
            continue

        print(f"  {split:<12} n={len(loader.dataset):>8}  p-score={metrics['pscore_mean']:.4f}")
        rows.append(
            {
                "archivo": spec.name,
                "dataset": spec.dataset,
                "modelo": Path(spec.model_cfg).stem,
                "seed": spec.seed,
                "non_iid_seed": seeds["non_iid_seed"],
                "split": split,
                "n_ejemplos": len(loader.dataset),
                "epoch_ckpt": epoch,
                "pscore_mean": metrics["pscore_mean"],
                "topsim": metrics["topsim"],
                "twonn_id": metrics["twonn_id"],
                "hoyer_sparsity": metrics["hoyer_sparsity"],
                "sv_auc": metrics["sv_auc"],
                "n_components_90pct": metrics["n_components_90pct"],
                "embedding_dim": metrics["embedding_dim"],
                # p-score registrado durante el entrenamiento, si lo hubo (split val_4cases)
                "pscore_train_log": best_ams.get("val_4cases_pscore_mean"),
            }
        )

    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

results = pd.DataFrame(rows)
results

## 10. Resumen y guardado

In [ ]:
if not results.empty:
    pivot = results.pivot_table(
        index=["dataset", "modelo", "seed"],
        columns="split",
        values="pscore_mean",
    )
    display(pivot)

    results.to_csv(RESULTS_CSV, index=False)
    print(f"\nGuardado en {RESULTS_CSV}")
else:
    print("Sin resultados.")

if failures:
    display(pd.DataFrame(failures))